In [ ]:
# Imports
import asyncio
import time
from typing import Literal
from enum import Enum, auto

# Dependencies
import pandas as pd

# Library
from utils.utils import remove_from_list
from pipeline.builder import VanguardPipeline
from data.vanguard_data import VanguardStorage
from parsers.normalizers import construct_rules
from parsers.vanguard_parser import VanguardParser
from api_builder.api_request import header, dict_construct
from classifier.vanguard_classifier import VanguardClassifier
from classifier.classifier import process_items, sort_storage_list
from api_builder.vanguard_api_build import MediaWikiAPI, VanguardScrapper
from fsm import	fsm, menus, parser
from fsm.fsm import State


In [ ]:
handlers = {
	State.ENTRY_POINT: menus.entry_point,
	State.SELECT_MAIN_CATEGORY: menus.select_category,
	State.SELECT_SUBCATEGORY: menus.select_subcategory,
	State.BUILD_QUERY: menus.make_query,
	State.FETCH: menus.fetch_routine,
}

In [ ]:
# Url Parser
url = None
state_machine = fsm.FSMContext()
state = State.ENTRY_POINT
while (state != State.END):
	if (state == State.ENTRY_POINT):
		state = menus.entry_point(state_machine)
	elif (state == State.SELECT_MAIN_CATEGORY):
		state = menus.select_category(state_machine)
	elif (state == State.SELECT_SUBCATEGORY):
		state = menus.select_subcategory(state_machine)
	elif (state == State.BUILD_QUERY):
		state = menus.make_query(state_machine)
	elif (state == State.FETCH):
		state = menus.fetch_routine(state_machine)

In [ ]:
state_machine.data

In [ ]:
rules = construct_rules("Technical")
web = MediaWikiAPI()
pipeline = VanguardPipeline(
	VanguardScrapper(web),
	VanguardParser(),
	VanguardClassifier(rules),
	VanguardStorage()
)
param = {
	"action": "parse",
	"page": "List of Cardfight!! Vanguard Technical Booster Sets",
	"format": "json"
}
await pipeline.scrapper.api.init_session()
links = await pipeline.scrapper.api.get(param, header)
await pipeline.scrapper.api.close_session()

In [ ]:
start_request = pipeline.scrapper.obtain_links(links)

In [ ]:
start_request

In [ ]:
str(param.values())

In [ ]:
pipeline.parser.clean_trash_from_set("List of Cardfight!! Vanguard Technical Booster Sets", start_request, 4)

In [ ]:
start_request

In [ ]:
process_items(start_request, pipeline.classifier, pipeline.storage)

In [ ]:
pipeline.storage.g

In [ ]:

async def request_api(param, header):
		api_answer = await pipeline.scrapper.api.get(param, header)
		return (api_answer)

web = MediaWikiAPI()
pipeline = VanguardPipeline(
	VanguardScrapper(web),
	VanguardParser(),
	VanguardClassifier(rules),
	VanguardStorage()
)

await pipeline.scrapper.api.init_session()
first_param = dict_construct('get', )
first_request =  await request_api(first_param, header)
await pipeline.scrapper.api.close_session()
start_request = pipeline.scrapper.obtain_links(first_request)
crude_consults = pipeline.parser.separate_urls(start_request)
consults = remove_from_list(crude_consults, [
	"Lyrical Monasterio", "The Mask Collection"
])
booster_requests = remove_from_list(start_request, [
	"G Booster Set 8: Collezione del Combattente Vol.1",
	"G Booster Set 9: Collezione del Combattente Vol.2",
	"Thailand Booster Set 1: The Mask Collection",
	"G Booster Set 7: Giudizio delle Lame Splendenti",
	"G Booster Set 1: Trascendenza Interdimensionale"
])
process_items(booster_requests, pipeline.classifier, pipeline.storage)
sort_storage_list(["LB", "G"], pipeline.classifier, pipeline.storage)
current_state = State.ENTRY_POINT
time.sleep(4)

start_message()
user_answer = input(
	"booster; special; deck; other: "
).lower()
parsed = parse_answer(user_answer)
while (parsed == None):
	print("Invalid option. Try again")
	user_answer = input(
		"booster; special; deck; other: "
	).lower()
	parsed = parse_answer(user_answer)
print("You selected:", parsed)
await pipeline.scrapper.api.init_session()

In [ ]:
consults

In [ ]:
map_functions = {
	"LB": pipeline.parser.make_consults(pipeline.storage.lb),
	"LL": pipeline.parser.make_consults(pipeline.storage.ll),
	"G": pipeline.parser.make_consults(pipeline.storage.g),
	"V": pipeline.parser.make_consults(pipeline.storage.v),
	"D": pipeline.parser.make_consults(pipeline.storage.d),
	"DZ": pipeline.parser.make_consults(pipeline.storage.dz)
}


In [ ]:
def	parse_by_answer():
	


await pipeline.scrapper.api.init_session()

async def request_api(param, header):
	async with semaphore:
		api_answer = await pipeline.scrapper.api.get(param, header)
		return (api_answer)

async def main():

	api_answer = await pipeline.scrapper.api.get(first_param, header)

	await asyncio.sleep(4)
	dz_consults = pipeline.parser.make_consults("consult", pipeline.storage.dz)
	api_answer = await pipeline.scrapper.api.get(params=dz_consults[0], headers=header)
	wikitex = pipeline.scrapper.obtain_wikitex(api_answer)
	data = pipeline.scrapper.make_cardlist_from_str(wikitex=wikitex)
	infobox = pipeline.parser.infobox(wikitex)
	rows = pipeline.storage.prepare_data(data, 6, is_d=True, infobox=infobox)
	df = pd.DataFrame(rows, columns=[
		"CardNo", "Name", "Grade", "Faction",
		"FactionType", "Type", "Rarity", "Release"
	])
	await pipeline.scrapper.api.close_session()
	return (df)

df = await (main())

In [ ]:
df.loc[df["Type"] == "Over"]

In [ ]:
api_answer = pipeline.scrapper.api.get(params=first_param, headers=header)
crude_data = pipeline.scrapper.obtain_links(api_answer)
crude_consults = pipeline.parser.separate_urls(crude_data)
crude_consults = remove_from_list(crude_consults, [
	"Lyrical Monasterio", "The Mask Collection"
])
consults = pipeline.parser.make_consults("get", crude_consults)
crude_data = remove_from_list(crude_data, [
	"G Booster Set 8: Collezione del Combattente Vol.1",
	"G Booster Set 9: Collezione del Combattente Vol.2",
	"Thailand Booster Set 1: The Mask Collection",
	"G Booster Set 7: Giudizio delle Lame Splendenti",
	"G Booster Set 1: Trascendenza Interdimensionale"
])
process_items(crude_data, pipeline.classifier, pipeline.storage)
sort_storage_list(["LB", "G"], pipeline.classifier, pipeline.storage)
time.sleep(2)
dz_consults = pipeline.parser.make_consults("consult", pipeline.storage.dz)
api_answer = pipeline.scrapper.api.get(params=dz_consults[0], headers=header)
wikitex = pipeline.scrapper.obtain_wikitex(api_answer)
data = pipeline.scrapper.make_cardlist_from_str(wikitex=wikitex)
infobox = pipeline.parser.infobox(wikitex)
rows = pipeline.storage.prepare_data(data, 6, is_d=True, infobox=infobox)
df = pd.DataFrame(rows, columns=[
	"CardNo", "Name", "Grade", "Faction",
	"FactionType", "Type", "Rarity", "Release"
])

In [ ]:
df.loc[10:30]

In [ ]:
curl = pipeline.scrapper.api.get(params=dz_consults[0], headers=header)
content = pipeline.scrapper.obtain_links(curl)
pipeline.parser.clean_trash_from_set(crude_consults, content, 4)
card_consult = pipeline.parser.make_consults("consult", content)
main_table = pipeline.scrapper.api.get(params=card_consult[0], headers=header)